### Step 1: Install Dependencies
Install the required packages for LangGraph and LangChain Ollama integration.

In [ ]:
!pip install -q langgraph langchain-core langchain-ollama

### Step 2: Configure Ollama Background Process
Launch the Ollama server service in the background and download the open-weight `gpt-oss` model.

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,701 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,605 kB]
Hit:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main am

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
!nohup ollama serve > ollama.log &

nohup: redirecting stderr to stdout


In [ ]:
import os, subprocess, time, requests, json, re
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)
print("Ollama server should be ready on http://localhost:11434")

Ollama server should be ready on http://localhost:11434


In [ ]:
!ollama pull gpt-oss

In [ ]:
!ollama pull gemma4:e2b

In [ ]:
!ollama pull llama3:8b

In [ ]:
!ollama pull gemma:7b

In [ ]:
!ollama pull qwen3.5:2b

In [ ]:
!ollama pull qwen2:0.5b

In [ ]:
!ollama pull qwen:0.5b

In [ ]:
!ollama pull deepseek-r1:1.5b

In [ ]:
OLLAMA_API_URL = "http://localhost:11434/api/generate"

def call_ollama(prompt, model="qwen2:0.5b", stream=False):
    payload = {"model": model, "prompt": prompt, "stream": stream}
    resp = requests.post(OLLAMA_API_URL, json=payload)
    if resp.status_code != 200:
        raise RuntimeError(f"Request failed: {resp.status_code}, {resp.text}")
    if stream:
        for line in resp.text.splitlines():
            if not line.strip():
                continue
            obj = json.loads(line)
            print(obj.get("response", ""), end="")
    else:
        data = resp.json()
        return data.get("response", "")

In [ ]:
import time

# CONFIGURATION & TEST CASES
# Using the model pulled in your environment
CHOSEN_MODEL = "qwen2:0.5b"

In [ ]:
stress_tests = {
    "Test 1: Core Strength (Semantic Summarization)": {
        "prompt": (
            "Summarize the following customer complaint into 1 sentence focusing only on the core issue: "
            "'Look, I've been a loyal customer for 5 years, but your new update completely broke the bluetooth sync. "
            "I tried restarting, clearing cache, updating my phone OS, and reinstalling the app. Nothing works. "
            "If this isn't fixed by Friday, I'm canceling my subscription and moving to your competitor.'"
        ),
        "expected_behavior": "Success. LLMs excel at processing semantic context, filtering noise, and extracting core intent."
    },

    "Test 2: Logic & Tokenization Failure (The Strawberry Problem)": {
        "prompt": "How many times does the letter 'r' appear in the word 'Strawberry'? Think step by step.",
        "expected_behavior": "Likely Failure/Struggle. Explains tokenization vs. character processing. LLMs process text as chunks (tokens), not individual letters."
    },

    "Test 3: Logic & Planning Riddle (Altered Classic)": {
        "prompt": (
            "A farmer needs to cross a river with a chicken and a bag of grain. "
            "The boat can only hold the farmer and one item at a time. "
            "If left alone, the chicken will eat the grain. "
            "How does the farmer get both across safely without anything being eaten? "
            "Note: The farmer does NOT have a wolf with him this time."
        ),
        "expected_behavior": "Potential Failure. Shows pattern-matching flaws. LLMs often regurgitate the classic 7-step 'Wolf, Goat, Cabbage' solution out of habit, even though this modified version only requires 2 simple trips."
    },

    "Test 4: Knowledge Cutoff & Hallucination": {
        "prompt": "What major breakthrough or event happened in world politics exactly three days ago? Provide details.",
        "expected_behavior": "Graceful refusal or Hallucination. Proves that static weights do not have real-time internet access, setting up the business case for RAG (Retrieval-Augmented Generation)."
    }
}

In [ ]:
# LIVE EXECUTION LOOP
print(f"STARTING LIVE STRESS TEST ON MODEL: {CHOSEN_MODEL}")


for test_name, test_data in stress_tests.items():
    print(f"\nRunning {test_name}...")
    print(f"Prompt Sent:\n\"{test_data['prompt']}\"")

    print("Model Response:")
    start_time = time.time()
    try:
        # Calling your custom ollama caller function
        response = call_ollama(test_data['prompt'], model=CHOSEN_MODEL, stream=False)
        print(response.strip())
    except Exception as e:
        print(f"Error communicating with Ollama: {e}")

    elapsed_time = time.time() - start_time
    print(f"Response Time: {elapsed_time:.2f} seconds")
    print(f"Note:\n   {test_data['expected_behavior']}")
    time.sleep(2) # Brief pause between live demos for student digest

STARTING LIVE STRESS TEST ON MODEL: qwen2:0.5b

Running Test 1: Core Strength (Semantic Summarization)...
Prompt Sent:
"Summarize the following customer complaint into 1 sentence focusing only on the core issue: 'Look, I've been a loyal customer for 5 years, but your new update completely broke the bluetooth sync. I tried restarting, clearing cache, updating my phone OS, and reinstalling the app. Nothing works. If this isn't fixed by Friday, I'm canceling my subscription and moving to your competitor.'"
Model Response:
After five years of using this product/service from your company, there has been a major issue with their update which broke bluetooth sync. The customer tried various approaches but it still did not work even after restarting, clearing cache, updating the OS, and reinstalling the app. If left unattended to, they fear that this malfunction will continue without being fixed on or before Friday.
Response Time: 2.44 seconds
Note:
   Success. LLMs excel at processing semanti